# 🖧 Modelos locales con Ollama

**Laboratorio de PLN — IFTS24**
Matías Barreto, 2026

**Encuentro 14 · Bloque 2 — 50 minutos**

---

## Objetivo

Correr un LLM 100% local usando Ollama — sin API key, sin costo, sin datos enviados a servidores externos.

## Al terminar este bloque vas a poder:

1. Instalar Ollama y descargar modelos desde la terminal.
2. Hacer inferencias de chat y extraer datos estructurados con el modelo local.
3. Comparar cuando conviene usar un modelo local vs una API cloud.

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **Ollama** | Herramienta open-source que empaqueta y ejecuta LLMs localmente con un comando. |
| **SLM (Small Language Model)** | Modelos compactos (<10B parámetros) optimizados para hardware doméstico. |
| **Gemma** | Familia de modelos open source de Google, basada en la misma tecnología que Gemini. |
| **System prompt** | Mensaje de configuración que define el comportamiento del modelo antes de la conversación. |
| **Quantización** | Técnica para reducir el tamaño del modelo sin perder mucha calidad. |

## Antes de empezar: instalá Ollama

Todo lo que tiene icono de terminal se ejecuta en la **terminal**, no en Jupyter.

```bash
# 1. Instalá Ollama desde https://ollama.com

# 2. Descargá el modelo (2B parámetros, ~1.4 GB)
ollama pull gemma:2b

# 3. Verificá que está disponible
ollama list
```

Con Ollama corriendo en segundo plano, las celdas de Python se conectan a el via `localhost:11434`.

In [ ]:
# Instalación de librerías
!uv pip install -q ollama pandas gradio

In [ ]:
# Importaciones
import json
import os
import pandas as pd
from ollama import chat

# Configuración del modelo
# Si descargaste otro modelo, cambiá este nombre
MODEL_NAME = "granite4:latest"  # Modelo pequeño y rápido
# MODEL_NAME = "gemma:7b"  # Modelo más potente
# MODEL_NAME = "llama2"  # Alternativa popular

print(f"Configuración completada")
print(f"Modelo a usar: {MODEL_NAME}")
print("\nVerificá que Ollama esté corriendo: ollama list")

## Parte 1 — Primera inferencia local

### Analogía

Ollama funciona como tener un chef privado en tu cocina: vos hacés el pedido, el lo prepara localmente, y el resultado nunca sale de tu casa. Nada viaja a un servidor externo.

### Dónde vive esto en el mundo real

Las empresas con datos sensibles (salud, legales, finanzas) usan modelos locales para que la información no salga de su infraestructura. Ollama democratiza eso: antes necesitabas un data center, ahora alcanza con una notebook decente.

In [ ]:
# Importamos la función 'chat' de la librería 'ollama' para comunicarnos con el modelo local
from ollama import chat

# Texto de prueba que contiene el comentario del cliente que queremos analizar
comentario = """
Fui al cine a ver la nueva de Marvel y la verdad que una decepción total.
La trama no tiene sentido, los efectos especiales están bien pero ya aburren.
Encima me cobraron una fortuna por la entrada y el pochoclo.
No la recomiendo para nada.
"""

# Creamos la lista de mensajes con el rol de usuario ('user') 
# y la consigna o prompt con las instrucciones de análisis en el contenido ('content')
mensaje = [
    {
        "role": "user",
        "content": f"Analizá el sentimiento de este comentario y respondé brevemente con una de las siguientes categorias 'NEU'(sentimiento neutral), 'NEG' (sentimiento negativo), 'POS' (sentimiento positivo):\n{comentario}"
    }
]

# Enviamos el mensaje al modelo local usando la función 'chat'.
# Usamos 'MODEL_NAME' (definido anteriormente en el notebook) para indicarle cuál modelo de IA debe usar
# y pasamos nuestra lista de mensajes en el parámetro 'messages'.
response = chat(model=MODEL_NAME, messages=mensaje)

# Extraemos la respuesta de texto generada por el modelo accediendo al atributo '.message.content'
respuesta = response.message.content

# Imprimimos la respuesta en pantalla para que el usuario pueda leerla directamente en la consola
print(respuesta)


In [ ]:
from IPython.display import display, HTML

# Definimos un diccionario que asocia cada categoría de sentimiento con su correspondiente color hexadecimal.
# Estos colores se usarán para formatear de forma amigable y visual el resultado final en la interfaz de Jupyter.
COLORES = {
    "POS": "#27ae60", # Verde para sentimientos Positivos
    "NEG": "#e74c3c", # Rojo para sentimientos Negativos
    "NEU": "#f39c12", # Naranja para sentimientos Neutrales
}

# Texto de prueba que contiene el comentario de un usuario sobre una película gastronómica o de entretenimiento.
comentario = """
Fui al cine a ver la nueva de Marvel y la verdad que una decepción total.
La trama no tiene sentido, los efectos especiales están bien pero ya aburren.
Encima me cobraron una fortuna por la entrada y el pochoclo.
No la recomiendo para nada.
"""

# Estructuramos el mensaje de entrada para el modelo en formato de conversación de chat.
# Cada elemento es un diccionario que representa el rol del participante ('user' o 'system') y su contenido escrito.
mensaje = [
    {
        "role": "user",
        "content": (
            f"Analizá el sentimiento de este comentario:\n{comentario}\n\n"
            "Respondé en este formato exacto, sin agregar nada más:\n"
            "CATEGORIA: POS | NEG | NEU\n"
            "RESUMEN: tres líneas explicando brevemente por qué."
        )
    }
]

# Llamamos a la función 'chat' de la librería 'ollama' para procesar nuestra consulta.
# Pasamos la variable 'MODEL_NAME' para indicarle cuál modelo local de LLM debe usar (como Gemma)
# y el parámetro 'messages' configurado con nuestra lista 'mensaje'.
response = chat(model=MODEL_NAME, messages=mensaje)

# Extraemos el contenido de texto plano de la respuesta del modelo accediendo a los atributos del objeto devuelto.
respuesta = response.message.content

# -----------------------------------------------------------------------------
# PASO 1: Extracción limpia de la categoría semántica
# -----------------------------------------------------------------------------

# Inicializamos la variable que contendrá la categoría detectada con un valor por defecto ("NEU")
categoria_detectada = "NEU"

# Separamos la respuesta completa en una lista de líneas individuales usando el método 'splitlines'
lineas_respuesta = respuesta.splitlines()

# Recorremos cada línea de la respuesta para buscar la clasificación
for linea in lineas_respuesta:
    # Removemos espacios adicionales y convertimos el texto a mayúsculas para evitar problemas de formato
    linea_limpia = linea.strip().upper()
    
    # Comprobamos si la línea actual comienza con la etiqueta esperada "CATEGORIA:"
    if linea_limpia.startswith("CATEGORIA:"):
        # Evaluamos cuál de las tres etiquetas válidas está contenida en la línea limpia
        if "POS" in linea_limpia:
            categoria_detectada = "POS"
        elif "NEG" in linea_limpia:
            categoria_detectada = "NEG"
        elif "NEU" in linea_limpia:
            categoria_detectada = "NEU"

# -----------------------------------------------------------------------------
# PASO 2: Extracción limpia del resumen explicativo
# -----------------------------------------------------------------------------

# Inicializamos la variable con el contenido completo por defecto
resumen_detectado = respuesta

# Convertimos la respuesta a minúsculas para realizar una búsqueda insensible a mayúsculas y minúsculas
respuesta_minusculas = respuesta.lower()

# Buscamos la posición del índice inicial de la etiqueta "resumen:" en el texto
posicion_resumen = respuesta_minusculas.find("resumen:")

# Si la etiqueta "resumen:" existe en el texto (es decir, el método 'find' no devuelve -1)
if posicion_resumen != -1:
    # Calculamos el punto de inicio del texto del resumen sumando el índice más la longitud de la palabra "resumen:" (8 caracteres)
    inicio_texto = posicion_resumen + 8
    
    # Recortamos la cadena de texto original (slicing) desde ese punto de inicio hasta el final de la respuesta
    texto_resumen = respuesta[inicio_texto:]
    
    # Removemos espacios en blanco iniciales y finales del texto recortado usando 'strip'
    resumen_detectado = texto_resumen.strip()

# -----------------------------------------------------------------------------
# PASO 3: Formateo y visualización en HTML
# -----------------------------------------------------------------------------

# Buscamos el color asociado a la categoría en nuestro diccionario de colores
color_visual = COLORES[categoria_detectada]

# Creamos una cadena de texto formateada con etiquetas HTML dinámicas utilizando un f-string de Python.
# Definimos el estilo inline para pintar el fondo de la categoría con el color correspondiente.
html_visual = f"""
<span style="background:{color_visual};color:white;padding:4px 14px;border-radius:6px;
             font-weight:bold;font-size:1.1em;">{categoria_detectada}</span>
<p style="margin-top:10px;">{resumen_detectado}</p>
"""

# Usamos la función 'display' e instanciamos la clase 'HTML' para renderizar e imprimir el código HTML en el notebook
display(HTML(html_visual))


### ✎ Para pensar

- Ollama corre en tu hardware. ¿Qué implica eso para la velocidad si tu computadora es vieja?
- ¿En qué casos seria inaceptable enviar datos a una API cloud? Pensá en al menos dos ejemplos concretos.

## Parte 2 — System prompt: definir el comportamiento

El **system prompt** es una instrucción que va antes de la conversación y define el caracter del modelo. La diferencia entre un asistente genérico y uno especializado está casi siempre en el system prompt.

Vas a ver el mismo prompt respondido con y sin system role para comparar.

### 🐛 Laboratorio de Rotura

El código de abajo hace dos llamadas con la *misma* pregunta pero una lleva system prompt y la otra no. Antes de ejecutarlo, predecí:

- ¿Cuánto cambia la respuesta?
- ¿En qué aspectos concretos esperás que difieran?

Ejecutá y comparalos con lo que predijiste.

- ¿Qué pasó en realidad?
- ¿Qué te dice eso sobre el poder del system prompt para definir comportamiento?

No mires la respuesta todavía.

In [ ]:
consulta = "¿Qué opinás de la última película de Tarantino?"

# SIN SYSTEM ROLE
print("="*80)
print("1. SIN SYSTEM ROLE (comportamiento genérico)")
print("="*80)

respuesta_generica = chat(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": consulta}]
)
print(respuesta_generica.message.content)

print("\n" + "="*80)
print("2. CON SYSTEM ROLE (crítico de cine especializado)")
print("="*80)

# CON SYSTEM ROLE
system_prompt_cine = """
Sos un crítico de cine argentino especializado en cine independiente y de autor.
Tenés 15 años de experiencia escribiendo para revistas culturales.
Tu estilo es:
- Análisis profundo de narrativa y dirección
- Referencias a cine clásico y contemporáneo
- Tono profesional pero accesible
- Comparaciones con otras obras del mismo director
Respondés en español rioplatense.
"""

respuesta_especializada = chat(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": system_prompt_cine},
        {"role": "user", "content": consulta}
    ]
)
print(respuesta_especializada.message.content)

### ✎ Para pensar

- ¿Qué información incluirías en el system prompt de un asistente para una empresa? Pensá en al menos 3 elementos.
- Si el system prompt dice 'respondé siempre en inglés' pero el usuario escribe en español, ¿qué debería hacer el modelo?

## Parte 3 — Extracción de datos estructurados

Una de las aplicaciones más valiosas: convertir texto no estructurado (reviews, correos, reportes) en JSON que el código puede procesar.

La clave es el system prompt: definís el schema y pedís **solo JSON, sin texto adicional**.

In [ ]:
def extract_json_from_text(text: str):
    """
    Extrae JSON de texto que puede contener explicaciones adicionales.
    Los modelos a veces agregan texto antes/después del JSON.
    """
    start = text.find('{')
    end = text.rfind('}')
    if start != -1 and end != -1 and end > start:
        candidate = text[start:end+1]
        try:
            return json.loads(candidate)
        except:
            return None
    return None

In [ ]:
# Review de un producto
review_producto = """
REVIEW: Auriculares Bluetooth XYZ-500
Comprador: Juan P. - Buenos Aires
Fecha: 15/03/2024

Los compré hace un mes y la verdad que re contentos. El sonido es excelente,
los graves se sienten mucho. La batería dura fácil 8 horas, perfecto para el laburo.
Lo único malo es que son medio pesados después de usarlos mucho rato, me cansan las orejas.
El precio me pareció razonable, 35 lucas, considerando la calidad que tienen.
La conexión Bluetooth es instantánea y nunca se corta.
Recomendación: 8/10, muy buenos para el precio.
"""

# System prompt para análisis estructurado
system_prompt_analisis = """
Sos un sistema de análisis de reviews que extrae información estructurada.
Tu tarea es convertir reviews en formato JSON con los siguientes campos:
- producto: nombre del producto
- comprador: nombre del comprador
- fecha: fecha de la review
- calificacion: número del 1 al 10
- aspectos_positivos: lista de aspectos positivos mencionados
- aspectos_negativos: lista de aspectos negativos mencionados
- precio_mencionado: precio si lo menciona, null si no
- sentimiento_general: "positivo", "negativo" o "neutral"

IMPORTANTE: Devolvé SOLO el JSON, sin explicaciones adicionales.
"""

# Prompt de extracción
prompt_extraccion = f"""
Extrae la información de esta review en formato JSON:

{review_producto}

Devolvé SOLO el JSON válido.
"""

# Procesamiento
print("Extrayendo datos estructurados de la review...")

response = chat(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": system_prompt_analisis},
        {"role": "user", "content": prompt_extraccion}
    ]
)

texto_respuesta = response.message.content
print("\nRespuesta del modelo:")
print(texto_respuesta)

# Extraer JSON
datos_estructurados = extract_json_from_text(texto_respuesta)

if datos_estructurados:
    print("\n" + "="*60)
    print("DATOS ESTRUCTURADOS EXTRAÍDOS:")
    print("="*60)
    print(json.dumps(datos_estructurados, indent=2, ensure_ascii=False))

    # Convertir a DataFrame
    df = pd.DataFrame([datos_estructurados])
    print("\nDataFrame generado:")
    print(df)
else:
    print("\nNo se pudo extraer JSON válido de la respuesta.")

In [ ]:
# --- Espacio de practica ---
#
# Crea tu propio extractor estructurado:
#   1. Elegi un tipo de texto (articulo, CV, descripcion de producto)
#   2. Define el schema JSON que queres extraer
#   3. Escribi el system prompt con ese schema
#   4. Proba con 2-3 ejemplos y verifica que el JSON sea valido
#
# ─────────────────────────────────────────────────────
MI_TEXTO = "Pega aca tu texto de ejemplo"

MI_SYSTEM_PROMPT = "Extraer la siguiente informacion en JSON: {campo: valor}. Devolver SOLO el JSON."

response = chat(
    model=MODEL_NAME,
    messages=[{"role": "system", "content": MI_SYSTEM_PROMPT}, {"role": "user", "content": MI_TEXTO}]
)
print(response.message.content)

## Parte 4 — Interfaz con Gradio

Gradio crea una interfaz web para el modelo en pocas líneas — útil para demostrar el modelo a usuarios no técnicos.

In [ ]:
import gradio as gr

def analizar_sentimiento(texto, detalle):
    """
    Analiza el sentimiento de un texto usando el modelo local.

    Parámetros:
    - texto: El texto a analizar
    - detalle: Nivel de detalle del análisis
    """
    if not texto.strip():
        return "Por favor ingresá un texto para analizar."

    # Configurar el prompt según el nivel de detalle
    if detalle == "Simple":
        prompt = f"""Analiza el sentimiento de este texto y respondé en una palabra:
        Positivo, Negativo o Neutral.

        Texto: {texto}"""
    else:
        prompt = f"""Analiza detalladamente el sentimiento de este texto.
        Incluye:
        - Sentimiento general (positivo/negativo/neutral)
        - Intensidad (baja/media/alta)
        - Emociones detectadas
        - Tono del texto

        Texto: {texto}"""

    try:
        response = chat(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.message.content
    except Exception as e:
        return f"Error: {str(e)}\n\nVerificá que Ollama esté corriendo."

# Crear la interfaz
with gr.Blocks(title="Analizador de Sentimiento Local") as demo:
    gr.Markdown("""
    # Analizador de Sentimiento con IA Local
    ### Procesamiento 100% privado en tu computadora

    Esta herramienta usa Ollama para analizar el sentimiento de cualquier texto
    sin enviar datos a servidores externos.
    """)

    with gr.Row():
        with gr.Column():
            texto_input = gr.Textbox(
                label="Texto a analizar",
                placeholder="Pegá acá el texto que querés analizar...",
                lines=5
            )

            detalle_input = gr.Radio(
                choices=["Simple", "Detallado"],
                value="Simple",
                label="Nivel de detalle"
            )

            btn_analizar = gr.Button(
                "Analizar Sentimiento",
                variant="primary"
            )

        with gr.Column():
            resultado_output = gr.Textbox(
                label="Resultado del análisis",
                lines=10
            )

    # Ejemplos predefinidos
    gr.Examples(
        examples=[
            ["La película estuvo increíble, la mejor que vi este año!", "Simple"],
            ["No me gustó para nada, perdí mi tiempo y mi plata.", "Detallado"],
            ["Estuvo bien, nada del otro mundo pero entretenida.", "Detallado"],
        ],
        inputs=[texto_input, detalle_input]
    )

    # Conectar botón con función
    btn_analizar.click(
        fn=analizar_sentimiento,
        inputs=[texto_input, detalle_input],
        outputs=resultado_output
    )

    gr.Markdown("""
    ---
    **Características de Privacidad:**
    - Todo el procesamiento se hace localmente
    - Ningún dato se envía a servidores externos
    - Funciona sin conexión a internet
    - Costo $0 en APIs
    """)

# Lanzar la interfaz
print("Iniciando interfaz web...")
print(f"Modelo: {MODEL_NAME}")
print("La interfaz se abrirá en tu navegador")

demo.launch(
    server_name="127.0.0.1",
    server_port=7860,
    share=False  # No compartir públicamente
)

## Local vs Cloud: cuándo usar cada uno

| Criterio | Ollama (local) | API cloud (Gemini / GPT) |
|---|---|---|
| **Privacidad** | Total — nada sale de tu máquina | Datos en servidores externos |
| **Costo** | $0 de API | Por token / por request |
| **Velocidad** | Depende de tu hardware | Generalmente más rápido |
| **Calidad** | Modelos más pequeños | Modelos mucho más potentes |
| **Internet** | No necesario | Obligatorio |

**Usá Ollama cuando:** privacidad o muchas consultas de bajo riesgo.

**Usá cloud cuando:** mejor calidad o hardware limitado.

## Cierre del bloque

| Concepto | Qué aprendiste |
|---|---|
| **Ollama** | Corre LLMs localmente con `ollama pull` + cliente Python |
| **System prompt** | Define el comportamiento del modelo antes de la conversación |
| **Extracción JSON** | System prompt con schema + pedir SOLO JSON |
| **Gradio** | Interface web para modelos en pocas líneas |
| **Local vs cloud** | Trade-offs entre privacidad, costo, calidad y velocidad |

**Próximo bloque →** Pydantic y humanidades digitales: cómo usar modelos locales para extraer datos estructurados de textos históricos.